# Tapi · Cómo reducir las consultas a DeltaPay

Analicé qué consultas podía evitar para acercarme al límite de 6 por pago por semana. Validé los datos, comparé frecuencias y revisé qué tenían en común las semanas que quedaron por encima del límite.

Los CSV se descargan desde GitHub.

In [ ]:
%pip -q install duckdb==1.4.3

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import duckdb

url_base = "https://raw.githubusercontent.com/DaianaSep/tapi_Challenge/main"
nombres = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}

carpeta = Path("data")
carpeta.mkdir(exist_ok=True)

nueva_conexion = duckdb.connect()
try:
    for tabla, archivo in nombres.items():
        destino = carpeta / archivo
        urlretrieve(f"{url_base}/{archivo}", destino)
        nueva_conexion.execute(
            f"CREATE TABLE {tabla} AS "
            "SELECT * FROM read_csv(?, header=true, all_varchar=true)",
            [str(destino)],
        )

except Exception:
    nueva_conexion.close()
    raise

if "conexion" in globals():
    conexion.close()
conexion = nueva_conexion

def consultar(sql):
    return conexion.execute(sql).df()

print("CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.")


CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.


Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Paso 1 · Preparar los datos e identificar los convenios

Primero conté los registros de cada tabla para dimensionar el análisis. Después comparé filas, IDs informados e IDs distintos para verificar si podía contar cuentas, consultas y pagos sin duplicarlos. Finalmente crucé convenios y proveedores para identificar qué pagos podía atribuir a DeltaPay.

In [2]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

,tabla,filas
0,cuentas,688
1,consultas,42907
2,pagos,2006


**Qué encontré.** Conté 688 registros en cuentas, 42.907 registros en consultas, 2.006 registros en pagos.

### Identificadores de cuentas

Comparé filas, IDs informados e IDs distintos para identificar cada cuenta a la que asignar una frecuencia.

In [3]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

,filas,filas_con_id,cuentas_distintas
0,688,688,688


**Qué encontré.** Las filas de cuentas coincidieron con los IDs informados y distintos.

### Identificadores de consultas

Comparé filas, IDs informados e IDs distintos para contar intentos sin duplicarlos.

In [4]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

,filas,filas_con_id,consultas_distintas
0,42907,42907,42907


**Qué encontré.** Las filas de consultas coincidieron con los IDs informados y distintos.

### Identificadores de pagos

Comparé filas, IDs informados e IDs distintos para contar pagos sin inflar el denominador.

In [5]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

,filas,filas_con_id,pagos_distintos
0,2006,2006,2006


**Qué encontré.** Las filas de pagos coincidieron con los IDs informados y distintos.

### Convenios y proveedores

Busqué qué convenios atendía DeltaPay y si compartían proveedor. Como el CSV de pagos no trae proveedor, este cruce justifica la atribución por convenio que uso para calcular el ratio.

In [6]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

,convenio,proveedor,cantidad_consultas
0,CONV-AGUA,deltapay,12337
1,CONV-CABLE,proveedor_c,4307
2,CONV-GAS,deltapay,4542
3,CONV-INTERNET,proveedor_c,1633
4,CONV-LUZ,deltapay,1529
5,CONV-TEL,deltapay,2836
6,CONV-TV,deltapay,15723


**Qué encontré.** En los datos, los convenios atendidos por DeltaPay aparecen con un solo proveedor. Usé esa relación para atribuir los pagos a DeltaPay.

## Paso 2 · Validar los datos antes de calcular el ratio

#### Qué tipo de consultas estaba contando

Agrupé los intentos por origen y resultado para distinguir las consultas programadas de los reintentos y ver cuáles encontraron deuda, cuáles devolvieron que no había deuda y cuáles fallaron. Cada fila del resultado resume intentos. Si una cuenta devolvió `deuda_encontrada` varias veces, cada consulta cuenta por separado.

In [7]:
consultar("""
SELECT origen, resultado, COUNT(*) AS intentos
FROM consultas
GROUP BY origen, resultado
ORDER BY origen, intentos DESC
""")

,origen,resultado,intentos
0,programada,sin_deuda,21523
1,programada,deuda_encontrada,11749
2,programada,timeout,4044
3,programada,error,365
4,reintento,sin_deuda,2835
5,reintento,deuda_encontrada,1428
6,reintento,timeout,927
7,reintento,error,36


**Qué encontré.** En el conjunto completo registré 37.681 consultas programadas y 5.226 reintentos. Hubo 4.971 `timeout` y 401 errores entre ambos orígenes. Para medir la carga conté los intentos por separado y, para seguir una factura, tomé las respuestas `deuda_encontrada`.

#### Qué período cubría cada archivo

Busqué la primera y la última fecha de alta, consulta y pago para entender qué período podía analizar con los tres archivos. En consultas y pagos tomé los primeros diez caracteres del timestamp para conservar la fecha local registrada. Esa revisión me ayudó a delimitar las semanas completas que comparé después.

In [8]:
consultar("""
SELECT 'cuentas' AS tabla, MIN(fecha_alta) AS desde, MAX(fecha_alta) AS hasta
FROM cuentas
UNION ALL
SELECT 'consultas', MIN(SUBSTR("timestamp", 1, 10)), MAX(SUBSTR("timestamp", 1, 10))
FROM consultas
UNION ALL
SELECT 'pagos', MIN(SUBSTR(fecha_pago, 1, 10)), MAX(SUBSTR(fecha_pago, 1, 10))
FROM pagos
""")

,tabla,desde,hasta
0,cuentas,2026-06-01,2026-07-15
1,consultas,2026-08-01,2026-10-31
2,pagos,2026-08-04,2026-10-31


**Qué encontré.** Las altas van del 1 de junio al 15 de julio. Las consultas abarcan del 1 de agosto al 31 de octubre y el primer pago observado fue el 4 de agosto. Para comparar el ratio usé las 12 semanas completas de lunes a domingo entre el 3 de agosto y el 25 de octubre. La fecha del primer pago observado cae dentro de la primera semana.

#### Si podía usar monto y vencimiento para seguir una factura

Conté los campos vacíos de monto y vencimiento para cada resultado de consulta. Cuando la respuesta fue `sin_deuda`, esperaba encontrar esos campos vacíos. Me interesaba especialmente `deuda_encontrada`, porque necesitaba el vencimiento para vincular la factura con el pago y el monto para detectar cambios entre consultas.

In [9]:
consultar("""
SELECT
    resultado,
    COUNT(*) AS intentos,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(monto), '') IS NULL) AS sin_monto,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(fecha_vencimiento), '') IS NULL) AS sin_vencimiento
FROM consultas
GROUP BY resultado
ORDER BY resultado
""")

,resultado,intentos,sin_monto,sin_vencimiento
0,deuda_encontrada,13177,0,0
1,error,401,401,401
2,sin_deuda,24358,24358,24358
3,timeout,4971,4971,4971


**Qué encontré.** Las 13.177 respuestas `deuda_encontrada` tenían monto y vencimiento informados. Ambos campos estaban vacíos en todas las respuestas `sin_deuda`, `timeout` y `error`. Esto me permitió usar el vencimiento para vincular las detecciones con los pagos y el monto para comparar respuestas sucesivas.

## Paso 3 · Medir el ratio semanal de DeltaPay

Conté todos los intentos de DeltaPay y los pagos atribuidos a sus convenios, agrupados por semana. Mostré ambos conteos para distinguir si el ratio alto venía de muchas consultas, de pocos pagos o de ambos. Usé las 12 semanas completas del 3 de agosto al 25 de octubre de 2026 y la fecha local del CSV.

In [10]:
consultar("""
WITH q AS (
    SELECT CAST(date_trunc('week', CAST(left(timestamp, 10) AS DATE)) AS DATE) AS semana,
           count(*) AS consultas
    FROM consultas WHERE proveedor = 'deltapay'
    GROUP BY 1
), p AS (
    SELECT CAST(date_trunc('week', CAST(left(fecha_pago, 10) AS DATE)) AS DATE) AS semana,
           count(*) AS pagos
    FROM pagos
    WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
    GROUP BY 1
)
SELECT semana, consultas, coalesce(pagos, 0) AS pagos,
       round(consultas * 1.0 / nullif(pagos, 0), 2) AS ratio
FROM q LEFT JOIN p USING (semana)
WHERE semana BETWEEN DATE '2026-08-03' AND DATE '2026-10-19'
ORDER BY semana
""")

,semana,consultas,pagos,ratio
0,2026-08-03,3046,8,380.75
1,2026-08-10,3641,131,27.79
2,2026-08-17,2633,169,15.58
3,2026-08-24,1824,139,13.12
4,2026-08-31,2905,40,72.63
5,2026-09-07,3405,76,44.80
6,2026-09-14,3415,172,19.85
7,2026-09-21,1944,160,12.15
8,2026-09-28,2813,76,37.01
9,2026-10-05,3290,42,78.33


**Qué encontré.** Las 12 semanas completas superaron el límite y el ratio del período fue de 25,76. En la semana del 3 de agosto hubo 3.046 consultas y solo 8 pagos. Ver ambos conteos me permitió identificar el efecto de las semanas con pocos pagos.

### Qué convenios concentran las consultas

Calculé qué porcentaje de las consultas de DeltaPay correspondía a cada convenio para dimensionar dónde estaba el volumen.

In [11]:
consultar("""
SELECT convenio, count(*) AS consultas,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS porcentaje
FROM consultas
WHERE proveedor = 'deltapay'
  AND left(timestamp, 10) BETWEEN '2026-08-03' AND '2026-10-25'
GROUP BY convenio ORDER BY consultas DESC
""")

,convenio,consultas,porcentaje
0,CONV-TV,14810,42.80
1,CONV-AGUA,11437,33.05
2,CONV-GAS,4227,12.22
3,CONV-TEL,2644,7.64
4,CONV-LUZ,1484,4.29


**Qué encontré.** TV y Agua concentraron el 75,9 % de las consultas. El volumen me sirvió para dimensionar el problema. Para elegir la frecuencia comparé también cuántas facturas seguían apareciendo en la simulación.

## Paso 4 · Identificar consultas que se pueden evitar

Después de medir el ratio, quise saber cuántas veces Tapi volvía a encontrar una factura que ya conocía. Vinculé las consultas exitosas con los pagos por cuenta, convenio y vencimiento. Para cada pago ordené las consultas anteriores, tomé la primera como punto de partida y comparé su importe con el de las siguientes.

In [12]:
consultar("""
WITH filtros AS (
    SELECT unnest([false, true]) AS exigir_monto_pago
), vinculos AS (
    SELECT exigir_monto_pago, p.pago_id, q.monto,
           row_number() OVER (PARTITION BY exigir_monto_pago, p.pago_id ORDER BY q.timestamp, q.consulta_id) AS orden,
           first_value(q.monto) OVER (PARTITION BY exigir_monto_pago, p.pago_id ORDER BY q.timestamp, q.consulta_id) AS monto_inicial
    FROM pagos p JOIN consultas q
      ON p.account_id = q.account_id AND p.convenio = q.convenio
     AND p.fecha_vencimiento = q.fecha_vencimiento
     AND q.timestamp < p.fecha_pago
    CROSS JOIN filtros
    WHERE (NOT exigir_monto_pago OR TRY_CAST(q.monto AS DECIMAL(18,2)) = TRY_CAST(p.monto AS DECIMAL(18,2)))
      AND q.proveedor = 'deltapay' AND q.resultado = 'deuda_encontrada'
), por_pago AS (
    SELECT exigir_monto_pago, pago_id, count(*) FILTER (WHERE orden > 1) AS reconsultas,
           count(*) FILTER (WHERE orden > 1 AND monto = monto_inicial) AS mismo_monto,
           count(*) FILTER (WHERE orden > 1 AND monto <> monto_inicial) AS cambio_monto
    FROM vinculos GROUP BY exigir_monto_pago, pago_id
)
SELECT exigir_monto_pago, count(*) AS pagos_vinculados,
       count(*) FILTER (WHERE reconsultas > 0) AS pagos_con_reconsulta,
       median(mismo_monto) FILTER (WHERE mismo_monto > 0) AS mediana_reconsultas_mismo_monto,
       count(*) FILTER (WHERE cambio_monto > 0) AS pagos_con_cambio_monto
FROM por_pago GROUP BY exigir_monto_pago ORDER BY exigir_monto_pago
""")

,exigir_monto_pago,pagos_vinculados,pagos_con_reconsulta,mediana_reconsultas_mismo_monto,pagos_con_cambio_monto
0,False,1432,642,12.0,93
1,True,1322,630,12.0,0


**Qué encontré.** Usé todo el archivo de pagos y comparé dos cruces. Al exigir que el importe consultado coincidiera con el pagado, **630 de 1.322 pagos vinculados (47,7 %)** tuvieron reconsultas, con una mediana de **12 reconsultas** entre esos casos. También hice el cruce por cuenta, convenio y vencimiento, que permitió identificar **93 pagos con algún cambio de importe** entre las respuestas previas. Ese resultado respaldó la idea de pausar las consultas de rutina tras detectar la deuda y confirmar el importe antes de pagar.

### Primera prueba de frecuencia en TV y Agua

Como TV y Agua concentraban el 75,9 % de las consultas, empecé por probar intervalos mínimos de 12 y 24 horas entre consultas programadas de una misma cuenta y convenio. Quería medir cuánto podía bajar el ratio con ese cambio. Conservé los reintentos y los pagos históricos para comparar las 12 semanas completas.

In [ ]:
import pandas as pd

consultas_delta = consultar("""
    SELECT account_id, convenio, "timestamp", origen, resultado, monto, fecha_vencimiento
    FROM consultas
    WHERE proveedor = 'deltapay'
    ORDER BY "timestamp", consulta_id
""")
pagos_delta = consultar("""
    SELECT account_id, convenio, fecha_pago
    FROM pagos
    WHERE convenio IN (
        SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
    )
""")

# Los CSV usan un único desplazamiento horario (-06:00).
from datetime import date, datetime

consultas_delta["timestamp"] = consultas_delta["timestamp"].map(datetime.fromisoformat)
consultas_delta["fecha_vencimiento"] = consultas_delta["fecha_vencimiento"].map(
    lambda valor: date.fromisoformat(valor) if valor is not None else None
)
pagos_delta["fecha_pago"] = pagos_delta["fecha_pago"].map(datetime.fromisoformat)
eventos = list(consultas_delta.itertuples(index=False))
pagos_observados = list(pagos_delta.itertuples(index=False))

from collections import Counter, defaultdict
from datetime import timedelta
from statistics import median

def medir_frecuencia(horas, horas_agua=None, horas_agua_despues=None):
    intervalos = {
        "CONV-TV": timedelta(hours=horas),
        "CONV-AGUA": timedelta(hours=horas if horas_agua is None else horas_agua),
    }
    objetivos = {"CONV-TV", "CONV-AGUA"}
    ultima_programada = {}
    primera_observada = {}
    primera_con_regla = {}
    resumen = defaultdict(Counter)

    def lunes(momento):
        dia = momento.date()
        return dia - timedelta(days=dia.weekday())

    for consulta in eventos:
        momento = consulta.timestamp
        cuenta = (consulta.account_id, consulta.convenio)
        semana = lunes(momento)
        resumen[semana]["consultas"] += 1
        omitir = False
        if consulta.convenio in objetivos and consulta.origen == "programada":
            anterior = ultima_programada.get(cuenta)
            minimo = intervalos[consulta.convenio]
            if consulta.convenio == "CONV-AGUA" and horas_agua_despues is not None and momento.day > 10:
                minimo = timedelta(hours=horas_agua_despues)
            omitir = anterior is not None and momento - anterior < minimo
            if not omitir:
                ultima_programada[cuenta] = momento

        if omitir:
            resumen[semana]["omitidas"] += 1

        if consulta.resultado == "deuda_encontrada" and consulta.fecha_vencimiento is not None:
            clave = (*cuenta, consulta.fecha_vencimiento)
            primera_observada.setdefault(clave, momento)
            if not omitir:
                primera_con_regla.setdefault(clave, momento)

    for pago in pagos_observados:
        resumen[lunes(pago.fecha_pago)]["pagos"] += 1

    semanas_completas = [
        v for semana, v in sorted(resumen.items())
        if date(2026, 8, 3) <= semana <= date(2026, 10, 19)
    ]
    demoras = []
    riesgos = Counter()
    for clave, original in primera_observada.items():
        if clave[1] not in objetivos or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
            continue
        siguiente = primera_con_regla.get(clave)
        if siguiente is None:
            riesgos["sin_deteccion"] += 1
        elif siguiente > original:
            demoras.append((siguiente - original).total_seconds() / 3600)
            if siguiente.date() >= clave[-1] and original.date() < clave[-1]:
                riesgos["pierde_margen_previo"] += 1

    return {
        "horas_tv": horas,
        "horas_agua": horas if horas_agua is None else horas_agua,
        "horas_agua_despues": horas_agua_despues,
        "consultas_omitidas": sum(v["omitidas"] for v in semanas_completas),
        "semanas_en_limite": sum(
            v["pagos"] > 0 and v["consultas"] - v["omitidas"] <= 6 * v["pagos"]
            for v in semanas_completas
        ),
        "mejor_ratio": min((v["consultas"] - v["omitidas"]) / v["pagos"] for v in semanas_completas if v["pagos"]),
        "detecciones_demoradas": len(demoras),
        "mediana_demora_horas": median(demoras) if demoras else None,
        "sin_deteccion_observada": riesgos["sin_deteccion"],
        "pasa_al_dia_de_vencimiento_o_despues": riesgos["pierde_margen_previo"],
    }


filas_primera_prueba = []
for horas in (12, 24):
    resultado = medir_frecuencia(horas)
    evitadas = resultado["consultas_omitidas"]
    filas_primera_prueba.append((horas, evitadas, round((34602 - evitadas) / 1343, 2),
                                round(resultado["mejor_ratio"], 2), resultado["semanas_en_limite"]))
pd.DataFrame(filas_primera_prueba, columns=["Intervalo mínimo (h)", "Consultas evitadas",
    "Ratio del período", "Mejor ratio semanal", "Semanas ≤ 6"])


Intervalo mínimo  Consultas evitadas  Ratio del período  Mejor ratio semanal  Semanas ≤ 6
            12 h               9.451              18,73                  9,96       0 de 12
            24 h              13.918              15,40                  8,05       0 de 12

**Qué encontré.** El ratio del período bajó de 25,76 a **18,73 con 12 h** y a **15,40 con 24 h**. El mejor ratio semanal fue de 9,96 y 8,05, respectivamente. En ambos casos, las 12 semanas siguieron por encima de 6. Por eso, en el paso 5 probé combinar intervalos más largos con una pausa cuando la deuda ya era conocida.

## Paso 5 · Simular intervalos de 24, 48 y 72 h

La idea fue espaciar la búsqueda y pausar la rutina cuando hay una factura conocida. Mantengo una consulta de respaldo cerca del vencimiento y contabilizo una revalidación hipotética antes de cada pago de los convenios modificados. Los pagos históricos se mantienen fijos. El replay conserva o descarta intentos observados: una factura ausente en las respuestas conservadas podría aparecer con otra programación real. La simulación puede conservar intentos históricos adicionales desde el día anterior al vencimiento.

In [13]:
import csv
from collections import Counter
from datetime import datetime, timedelta, date
from pathlib import Path

def simular_estado(consultas, pagos, intervalo, ventanas=None, reintento_dias=()):
    eventos = [(datetime.fromisoformat(r['timestamp']), 1, r['consulta_id'], r) for r in consultas]
    eventos += [(datetime.fromisoformat(r['fecha_pago']), 0, r['pago_id'], r) for r in pagos]
    estado, primera, retenida = {}, {}, {}
    guardadas=[]
    pendientes=set()
    semanal = {}
    for t, tipo, _, r in sorted(eventos):
        dia = t.date()
        lunes = dia - timedelta(days=dia.weekday())
        m = semanal.setdefault(lunes, Counter())
        k = r['account_id'], r['convenio']
        if tipo == 0:
            m['pagos'] += 1
            m['revalidaciones'] += 1
            estado.pop(k, None)
            pendientes.discard(k)
            continue
        m['observadas'] += 1
        deuda = r['resultado'] == 'deuda_encontrada' and r['fecha_vencimiento']
        if deuda:
            clave = (*k, date.fromisoformat(r['fecha_vencimiento']))
            primera.setdefault(clave, t)
        ultima, vence = estado.get(k, (None, None))
        horas = intervalo
        ventana = (ventanas or {}).get(k)
        if dia.month == 10 and not vence and ventana and not ventana[0] <= dia.day <= ventana[1]:
            horas = 96
        es_reintento = k in pendientes and ultima is not None and t-ultima < timedelta(hours=horas)
        if not es_reintento and ultima is not None and (t-ultima < timedelta(hours=horas) or (vence and dia < vence-timedelta(days=1))):
            # El respaldo prevalece cuando falta un día para el vencimiento conocido.
            if not vence or dia < vence-timedelta(days=1):
                continue
        m['ejecutadas'] += 1
        guardadas.append(r)
        pendientes.discard(k)
        if not es_reintento and r['resultado'] in ('timeout','error') and vence is None and dia.day in reintento_dias:
            pendientes.add(k)
        if deuda:
            retenida.setdefault(clave, t)
        nuevo_vence = clave[-1] if deuda else vence if r['resultado'] in ('timeout','error') else None
        estado[k] = (ultima if es_reintento else t, nuevo_vence)
    return semanal, primera, retenida, guardadas

### ¿Qué ahorro y qué riesgo trae cada frecuencia?

Comparé el volumen, las semanas dentro del límite y las facturas del histórico que ya no aparecen en las respuestas conservadas. Así puedo ver si ahorrar más consultas cambia el cumplimiento semanal.

In [14]:
def leer(nombre):
    with (Path("data") / nombres[nombre]).open() as archivo:
        return list(csv.DictReader(archivo))

consultas_delta = [r for r in leer("consultas") if r["proveedor"] == "deltapay"]
convenios = sorted({r["convenio"] for r in consultas_delta})
pagos_delta = [r for r in leer("pagos") if r["convenio"] in convenios]

def evaluar(intervalo, luz_historica=True, tv=None, ventanas=None, reintento_dias=()):
    consultas_semana, pagos_semana = Counter(), Counter()
    primeras, detecciones = {}, {}
    for convenio in convenios:
        cc = [r for r in consultas_delta if r["convenio"] == convenio]
        pp = [r for r in pagos_delta if r["convenio"] == convenio]
        horas = tv if convenio == "CONV-TV" and tv is not None else intervalo
        dias = reintento_dias if convenio == "CONV-TV" else ()
        semanas, original, retenida, _ = simular_estado(cc, pp, horas, ventanas, dias)
        sin_cambios = luz_historica and convenio == "CONV-LUZ"
        primeras.update(original)
        detecciones.update(original if sin_cambios else retenida)
        for semana, datos in semanas.items():
            consultas_semana[semana] += datos["observadas"] if sin_cambios else datos["ejecutadas"] + datos["revalidaciones"]
            pagos_semana[semana] += datos["pagos"]
    return consultas_semana, pagos_semana, primeras, detecciones

def resumir(resultado, desde, hasta, desde_facturas, hasta_facturas):
    consultas, pagos, primeras, detecciones = resultado
    semanas = [s for s in sorted(pagos) if desde <= s <= hasta]
    q, p = sum(consultas[s] for s in semanas), sum(pagos[s] for s in semanas)
    ausentes = Counter(k[1] for k,t in primeras.items()
                       if desde_facturas <= t.date() <= hasta_facturas and k not in detecciones)
    return q, round(q/p, 2), sum(0 < pagos[s] and consultas[s] <= 6*pagos[s] for s in semanas), dict(ausentes)

base = evaluar(48)
periodo = (date(2026,8,3), date(2026,10,19), date(2026,8,3), date(2026,10,25))
escenarios = {
    "24 h para todos": evaluar(24, luz_historica=False),
    "72 h para todos": evaluar(72, luz_historica=False),
    "48 h para todos": evaluar(48, luz_historica=False),
    "48 h salvo Luz": base,
    "72 h salvo Luz": evaluar(72),
    "96 h salvo Luz": evaluar(96),
    "TV 24 h": evaluar(48, tv=24),
    "TV 24 h y reintento cualquier día": evaluar(48, tv=24, reintento_dias=range(1,32)),
    "TV 24 h y reintento en ventanas": evaluar(48, tv=24, reintento_dias={5,6,7,20,21,22,23,24}),
}
import pandas as pd
pd.DataFrame({nombre: resumir(r, *periodo) for nombre,r in escenarios.items()},
             index=["Consultas", "Ratio", "Semanas ≤ 6", "Facturas ausentes por convenio"]).T

,Consultas,Ratio,Semanas ≤ 6,Facturas ausentes por convenio
24 h para todos,9628,7.17,6,"{'CONV-LUZ': 363, 'CONV-TV': 3}"
72 h para todos,5703,4.25,7,"{'CONV-LUZ': 724, 'CONV-TV': 15}"
48 h para todos,6722,5.01,7,"{'CONV-LUZ': 724, 'CONV-TV': 7}"
48 h salvo Luz,6739,5.02,7,{'CONV-TV': 7}
72 h salvo Luz,5720,4.26,7,{'CONV-TV': 15}
96 h salvo Luz,5183,3.86,8,"{'CONV-AGUA': 2, 'CONV-TV': 20}"
TV 24 h,7886,5.87,7,{'CONV-TV': 3}
TV 24 h y reintento cualquier día,8265,6.15,6,{}
TV 24 h y reintento en ventanas,7966,5.93,7,{}


**Qué encontré.** Con 24 h para todos, el ratio del período fue de 7,17 y cumplieron 6 de 12 semanas. Con 48 h y 72 h, cumplieron 7 de 12 en ambos casos. A 48 h, 724 de las 731 facturas ausentes eran de Luz. Por eso conservé su programación por cuenta, que habitualmente agrupa dos consultas en días consecutivos cerca de una vez al mes.

Con esa excepción, **48 h dejó el ratio en 5,02 y siete facturas de TV fuera de las consultas conservadas**. Extender a 72 h mantuvo las mismas siete semanas dentro del límite y elevó los casos ausentes a 15. A 96 h sumé una semana, con 22 casos ausentes. 

### Cuánto margen queda antes del vencimiento

Quise comprobar si espaciar las consultas dejaba tiempo para pagar. Medí los días calendario entre la primera detección y el vencimiento para cada convenio. Tomé las facturas que aparecieron por primera vez durante las 12 semanas completas y comparé el histórico, 48 h para todos y 48 h manteniendo la programación de Luz. Conté aparte las facturas que nunca aparecieron en las consultas conservadas, porque su margen en ese escenario es desconocido. Usé dos días como corte descriptivo para comparar escenarios.

In [ ]:
from statistics import median

escenarios_margen = {
    "Histórico": base[2],
    "48 h salvo Luz": base[3],
    "48 h para todos": escenarios["48 h para todos"][3],
}
filas_margen = []
for convenio in convenios:
    facturas = {
        clave: primera for clave, primera in base[2].items()
        if clave[1] == convenio and periodo[2] <= primera.date() <= periodo[3]
    }
    for nombre, detecciones in escenarios_margen.items():
        margen = [(clave[2] - detecciones[clave].date()).days
                  for clave in facturas if clave in detecciones]
        filas_margen.append({
            "Convenio": convenio, "Escenario": nombre, "Facturas históricas": len(facturas),
            "Mediana de días de margen": median(margen) if margen else None,
            "Con ≥ 2 días": sum(d >= 2 for d in margen),
            "Con 1 día": margen.count(1),
            "El día del vencimiento": margen.count(0),
            "Después del vencimiento": sum(d < 0 for d in margen),
            "Sin detección": len(facturas) - len(margen),
        })

margenes_por_convenio = pd.DataFrame(filas_margen)
display(margenes_por_convenio)

# Margen histórico de Luz en las facturas ausentes al aplicar 48 h a todos.
luz_ausentes = [
    (clave[2] - primera.date()).days
    for clave, primera in base[2].items()
    if clave[1] == "CONV-LUZ" and periodo[2] <= primera.date() <= periodo[3]
    and clave not in escenarios_margen["48 h para todos"]
]
print(f"Luz: {len(luz_ausentes)} ausentes con 48 h para todos; "
      f"{sum(d >= 2 for d in luz_ausentes)} tenían al menos 2 días de margen histórico; "
      f"mediana histórica: {median(luz_ausentes)} días.")

 Convenio       Escenario  Facturas históricas  Mediana de días de margen  Con ≥ 2 días  Con 1 día  El día del vencimiento  Después del vencimiento  Sin detección
CONV-AGUA       Histórico                  301                       15.0           301          0                       0                        0              0
CONV-AGUA  48 h salvo Luz                  301                       13.0           301          0                       0                        0              0
CONV-AGUA 48 h para todos                  301                       13.0           301          0                       0                        0              0
 CONV-GAS       Histórico                   35                       17.0            35          0                       0                        0              0
 CONV-GAS  48 h salvo Luz                   35                       15.0            35          0                       0                        0              0
 CONV-GAS 48 h para to

**Qué encontré.** Con 48 h manteniendo la programación de Luz, todas las facturas que aparecieron en la simulación se detectaron al menos dos días calendario antes del vencimiento. La mediana pasó de 15 a 13 días en Agua, de 17 a 15 en Gas, de 19 a 18 en Teléfono y de 11 a 10 en TV. Luz quedó en 12 días. Siete facturas de TV quedaron sin detección en las consultas conservadas.

En Luz, las 724 facturas que quedaron sin detección con 48 h para todos tenían al menos dos días de margen en su primera detección histórica, con una mediana de 12 días. Ese margen justifica probar otra programación para Luz. Como la simulación solo conserva o descarta consultas ya realizadas, queda por comprobar si una consulta nueva al día siguiente o a las 48 h habría encontrado la factura con tiempo para pagar.

**Qué margen queda si repito la búsqueda.** Miré el mínimo, además de la mediana, para ver los casos más ajustados. Estos son días calendario desde la primera detección en cada escenario. Restar uno o dos días es una estimación del margen si una consulta posterior también encuentra la factura.

| Convenio | Mínimo histórico | Mínimo con 48 h salvo Luz | Si demoro 1 día | Si demoro 2 días |
|---|---:|---:|---:|---:|
| Agua | 10 | 6 | 5 | 4 |
| Gas | 13 | 11 | 10 | 9 |
| Luz | 7 | 7 | 6 | 5 |
| Teléfono | 15 | 9 | 8 | 7 |
| TV | 6 | 2 | 1 | 0 |

En Luz, incluso las 724 facturas que no reaparecieron al aplicar 48 h a todos tenían al menos **7 días de margen histórico**. Si una nueva consulta las encontrara dos días después de la primera detección histórica, todavía quedarían al menos **5 días calendario**. En TV, tres facturas detectadas con la regla de 48 h tenían justo dos días de margen. Si demorara dos días más, quedarían para el día del vencimiento. Las siete facturas de TV que no reaparecieron en las consultas conservadas quedan fuera de ese cálculo.

## Paso 6 · Elegir la regla para el piloto

La mejor opción para el piloto es buscar facturas cada **48 h**, mantener la programación de Luz y pausar las consultas de rutina cuando Tapi ya conoce la deuda. Confirmaría el importe antes de pagar.

El ratio del período bajó de **25,76 a 5,02** y **7 de 12 semanas** quedaron dentro del límite. Con 72 h se mantuvieron las mismas siete semanas y más facturas dejaron de aparecer en las consultas conservadas. Las siete facturas ausentes con 48 h eran de TV. Si el patrón se repite en un mes nuevo, probaría acortar a **24 h solo en TV** y mediría si mejora la detección con tiempo para pagar.

## Paso 7 · Revisar las semanas que superaron el límite

Para las 12 semanas completas comparé el volumen de pagos y consultas. Calculé cuánto habría que recortar en cada semana para quedar en seis consultas por pago.

In [ ]:
consultas_48h, pagos_por_semana, _, _ = base

# Semanas completas de todo el período.
filas_brecha = []
for semana, pagos_semana in sorted(pagos_por_semana.items()):
    if not date(2026, 8, 3) <= semana <= date(2026, 10, 19):
        continue
    consultas_semana = consultas_48h[semana]
    consultas_luz = sum(
        date.fromisoformat(r["timestamp"][:10]) >= semana
        and date.fromisoformat(r["timestamp"][:10]) < semana + timedelta(days=7)
        for r in consultas_delta if r["convenio"] == "CONV-LUZ"
    )
    filas_brecha.append((semana, pagos_semana, consultas_semana, consultas_luz,
                         consultas_semana - 6 * pagos_semana))
brechas = pd.DataFrame(filas_brecha, columns=["Semana", "Pagos", "Consultas con 48 h",
                                            "Consultas de Luz", "Brecha sobre el límite"])
brechas["Ratio con 48 h"] = (brechas["Consultas con 48 h"] / brechas["Pagos"]).round(2)
fuera_limite = brechas["Brecha sobre el límite"] > 0
display(brechas.loc[fuera_limite])
display(brechas.groupby(fuera_limite.rename("Supera 6"))[
    ["Pagos", "Consultas con 48 h"]
].agg(["min", "median", "max"]))

**Qué encontré.** Las cinco semanas fuera del límite tuvieron entre **8 y 76 pagos** (mediana **42**), frente a **131 a 172** (mediana **163**) en las siete que cumplieron. Las consultas simuladas se solaparon entre ambos grupos. La mediana fue **601** en las semanas fuera del límite y **553** en las que cumplieron. En estos datos, el bajo volumen de pagos fue la diferencia más marcada.

La mayor brecha fue la semana del 3 de agosto. Para cumplir con 8 pagos habría que bajar de 611 a 48 consultas. La menor fue la del 28 de septiembre, que necesitaba recortar 145 consultas.

## Paso 8 · Buscar un patrón en las semanas con pocos pagos

Comparé los días 1 al 7 de cada mes para ver si el bajo volumen al inicio del mes se repetía.

In [ ]:
# Comparo ventanas idénticas del calendario, aparte de las semanas lunes a domingo.
pagos_por_mes = Counter()
pagos_del_1_al_7 = Counter()
for pago in pagos_delta:
    dia = date.fromisoformat(pago["fecha_pago"][:10])
    mes = dia.strftime("%Y-%m")
    pagos_por_mes[mes] += 1
    if dia.day <= 7:
        pagos_del_1_al_7[mes] += 1
display(pd.DataFrame([
    {"Mes": mes, "Pagos del 1 al 7": pagos_del_1_al_7[mes], "Pagos del mes": total,
     "% del 1 al 7": round(100 * pagos_del_1_al_7[mes] / total, 1)}
    for mes, total in sorted(pagos_por_mes.items())
]))

**Qué encontré.** Vi que las cinco semanas que superaron el límite coincidieron con el inicio o el cambio de mes. Para ver si el bajo volumen se repetía, miré los pagos del día 1 al 7. Encontré **2 en agosto, 24 en septiembre y 34 en octubre**, entre **0,4 % y 6,9 %** de los pagos de cada mes. En los tres meses, la mayor parte de los pagos se hizo después del día 7. Como tengo solo tres meses de datos y el primer pago registrado en agosto fue el día 4, tomaría este patrón como una hipótesis para validar con más meses.

**Qué probaría.** Mantendría 48 h como punto de partida y simularía un presupuesto semanal de consultas a partir de los pagos proyectados y confirmados, sin usar pagos futuros para decidir. Cuando se proyecten pocos pagos, reservaría capacidad para las revalidaciones antes de pagar y las deudas próximas a vencer. Diferiría primero las búsquedas de rutina menos urgentes y mediría si las facturas siguen detectándose con margen para pagar.

El 3 de agosto muestra un límite de esa idea. Con **8 pagos**, el máximo era de **48 consultas**, y **Luz sola sumó 204**. Para semanas así, discutiría con DeltaPay cómo aplicar el límite cuando hay muy pocos pagos, por ejemplo mediante un mínimo de consultas permitido o una ventana de medición más larga.